In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/datasets/weather_dataset.csv')

In [ ]:
df.head()

,PM2.5,PM10,temperature,humidity,wind_speed,atmospheric_pressure,historical_pollution,weather
0,34.418456,100.454965,30.483571,54.822579,5.394872,1008.670423,41.539665,Rainy
1,28.759814,53.302591,27.308678,60.417508,5.149614,1014.039269,61.304593,Sunny
2,13.450390,40.666801,31.238443,56.039284,0.970442,1008.547496,77.044040,Rainy
3,45.009132,79.534923,35.615149,66.656271,2.391847,1008.754100,46.103422,Rainy
4,29.649954,28.124185,26.829233,82.957678,3.463133,994.753498,38.668372,Storm


In [ ]:
print(df["weather"].value_counts())

weather
Sunny     2963
Storm     2691
Rainy     1631
Normal    1440
Windy     1275
Name: count, dtype: int64


In [ ]:
X = df[
    [
        "temperature",
        "humidity",
        "wind_speed",
        "atmospheric_pressure"
    ]
]

In [ ]:
y = df["weather"]

In [ ]:
X.head()

,temperature,humidity,wind_speed,atmospheric_pressure
0,30.483571,54.822579,5.394872,1008.670423
1,27.308678,60.417508,5.149614,1014.039269
2,31.238443,56.039284,0.970442,1008.547496
3,35.615149,66.656271,2.391847,1008.754100
4,26.829233,82.957678,3.463133,994.753498


In [ ]:
y.head()

,weather
0,Rainy
1,Sunny
2,Rainy
3,Rainy
4,Storm


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
y_train.shape

(8000,)

In [ ]:
y_test.shape

(2000,)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier # Import RandomForestClassifier

In [ ]:
# Create a pipeline that first scales the data, then applies RandomForestClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)) # Using RandomForestClassifier
])

print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 RandomForestClassifier(max_depth=10, random_state=42))])


In [ ]:
# Train the pipeline
pipeline.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


Now that the model is trained, the next step in the workflow is to make predictions using the test data (`X_test`) and then evaluate the model's performance.

In [ ]:
# Make predictions on the test set
y_pred = pipeline.predict(X_test)
print("Predictions generated.")

Predictions generated.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (weighted): {precision:.4f}")
print(f"Recall (weighted): {recall:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7695
Precision (weighted): 0.7697
Recall (weighted): 0.7695
F1-Score (weighted): 0.7689

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.62      0.63       288
       Rainy       0.67      0.72      0.69       326
       Storm       0.82      0.83      0.82       538
       Sunny       0.84      0.86      0.85       593
       Windy       0.76      0.66      0.71       255

    accuracy                           0.77      2000
   macro avg       0.75      0.74      0.74      2000
weighted avg       0.77      0.77      0.77      2000



In [ ]:
import pickle

# Define the filename for your pickled pipeline
pipeline_filename = '/content/drive/MyDrive/weather_prediction_pipeline.pkl'

# Save the pipeline to a .pkl file
with open(pipeline_filename, 'wb') as file:
    pickle.dump(pipeline, file)

print(f"Pipeline successfully exported to {pipeline_filename}")

Pipeline successfully exported to /content/drive/MyDrive/weather_prediction_pipeline.pkl


### Hyperparameter Tuning for Random Forest Classifier

To further optimize the Random Forest Classifier, we will perform hyperparameter tuning using `GridSearchCV`. This process explores different combinations of predefined hyperparameters and evaluates their performance to find the best set for our model.

We will tune the following parameters:
- `n_estimators`: The number of trees in the forest.
- `max_features`: The number of features to consider when looking for the best split.
- `max_depth`: The maximum depth of the tree.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'classifier__n_estimators': [50, 100, 200],  # Number of trees in the forest
    'classifier__max_features': ['sqrt', 'log2'], # Number of features to consider at each split
    'classifier__max_depth': [5, 10, 15]          # Maximum depth of the tree
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

print("GridSearchCV initialized.")

GridSearchCV initialized.


In [ ]:
# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

print("Grid search complete.")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Grid search complete.


In [ ]:
# Print the best parameters and best score
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

Best parameters found: {'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__n_estimators': 200}
Best cross-validation score: 0.7664


In [ ]:
# Get the best estimator (the tuned pipeline)
best_rf_model = grid_search.best_estimator_

# Make predictions on the test set using the best model
y_pred_tuned = best_rf_model.predict(X_test)

print("Predictions generated with the tuned model.")

Predictions generated with the tuned model.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Calculate evaluation metrics for the tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, average='weighted')
recall_tuned = recall_score(y_test, y_pred_tuned, average='weighted')
f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted')

print(f"Tuned Model Accuracy: {accuracy_tuned:.4f}")
print(f"Tuned Model Precision (weighted): {precision_tuned:.4f}")
print(f"Tuned Model Recall (weighted): {recall_tuned:.4f}")
print(f"Tuned Model F1-Score (weighted): {f1_tuned:.4f}")

print("\nTuned Model Classification Report:")
print(classification_report(y_test, y_pred_tuned))

Tuned Model Accuracy: 0.7665
Tuned Model Precision (weighted): 0.7668
Tuned Model Recall (weighted): 0.7665
Tuned Model F1-Score (weighted): 0.7660

Tuned Model Classification Report:
              precision    recall  f1-score   support

      Normal       0.64      0.62      0.63       288
       Rainy       0.65      0.70      0.68       326
       Storm       0.82      0.83      0.82       538
       Sunny       0.85      0.86      0.85       593
       Windy       0.77      0.67      0.71       255

    accuracy                           0.77      2000
   macro avg       0.74      0.74      0.74      2000
weighted avg       0.77      0.77      0.77      2000

